# B2：Text-to-SQL Agent — 安全架構設計

> **場景：** 台灣連鎖零售業（全台 50 家門市），讓非技術的門市主管用自然語言查詢銷售資料，不需要寫 SQL。  
> **核心挑戰：** 不能讓 LLM 生成危險的 SQL（DROP、DELETE、UPDATE），也不能讓一個門市的人查到其他門市的資料。

## 核心架構決策

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| SQL 安全 | 白名單 + AST 解析 | 只靠 Prompt 說不要做危險操作 |
| 資料隔離 | Row-Level Security | 讓 LLM 自己加 WHERE 子句 |
| 錯誤恢復 | 自動修正迴圈（max 2） | 讓用戶看到 SQL 錯誤 |
| Schema 注入 | 動態 Schema + 範例 | 靜態完整 Schema（太長） |
| 資料庫連接 | Read-only 帳號 | 共用帳號 |

In [ ]:
import re
import json
import sqlite3
import asyncio
from typing import Optional
from dataclasses import dataclass

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬 SQL")

print("環境就緒")

## 建立測試資料庫

In [ ]:
# 建立 SQLite 資料庫（模擬生產 BigQuery/CloudSQL）
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE stores (
    store_id TEXT PRIMARY KEY,
    store_name TEXT,
    region TEXT
);
INSERT INTO stores VALUES ('S001', '台北信義店', '北部');
INSERT INTO stores VALUES ('S002', '台中旗艦店', '中部');
INSERT INTO stores VALUES ('S003', '高雄三多店', '南部');

CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    store_id TEXT,
    product_name TEXT,
    quantity INTEGER,
    unit_price REAL,
    sale_date TEXT
);
INSERT INTO sales VALUES (1, 'S001', '有機牛奶', 150, 85.0, '2026-06-01');
INSERT INTO sales VALUES (2, 'S001', '全麥吐司', 200, 45.0, '2026-06-01');
INSERT INTO sales VALUES (3, 'S002', '有機牛奶', 80,  85.0, '2026-06-01');
INSERT INTO sales VALUES (4, 'S003', '蘋果汁',   120, 35.0, '2026-06-01');
INSERT INTO sales VALUES (5, 'S001', '有機牛奶', 180, 85.0, '2026-06-02');
""")
conn.commit()

SCHEMA = """
Tables:
  stores(store_id TEXT PK, store_name TEXT, region TEXT)
  sales(sale_id INT PK, store_id TEXT FK→stores, product_name TEXT,
        quantity INT, unit_price REAL, sale_date TEXT)

Important:
  - sale_date format: 'YYYY-MM-DD'
  - Revenue = quantity * unit_price
  - Always filter by store_id for data isolation
"""

print("✅ 測試資料庫建立完成")
cursor.execute("SELECT COUNT(*) FROM sales")
print(f"   sales 資料筆數：{cursor.fetchone()[0]}")

## 決策 1：SQL 安全驗證 — 為什麼不只靠 Prompt？

**❌ 被拒絕：只在 System Prompt 說「不要生成危險 SQL」**  
LLM 有時候會「忘記」指令，特別是在複雜的多步驟查詢中。  
攻擊者可以用 Prompt Injection 繞過 system prompt 的限制。

**✅ 選擇：程式碼層的 SQL 驗證（不依賴 LLM 的「好行為」）**

In [ ]:
class SQLSafetyValidator:
    """
    SQL 安全驗證器（在執行前攔截）
    
    為什麼要程式碼層驗證，不靠 Prompt？
    → Prompt 是建議，程式碼是強制。安全不能靠建議。
    
    為什麼用 Regex + 關鍵字，不用 SQL Parser（如 sqlfluff）？
    → 對於 Read-only 驗證，關鍵字夠用，Parser 引入依賴和複雜度
    → 如果需要更細粒度（如允許 CREATE TEMP TABLE），才考慮 Parser
    """
    
    # 危險操作關鍵字（這些操作 Read-only 用戶不應該能執行）
    DANGEROUS_KEYWORDS = [
        r'\bDROP\b', r'\bDELETE\b', r'\bUPDATE\b', r'\bINSERT\b',
        r'\bALTER\b', r'\bTRUNCATE\b', r'\bCREATE\b', r'\bGRANT\b',
        r'\bREVOKE\b', r'\bEXEC\b', r'\bEXECUTE\b', r'\bSP_\b',
        r'\bXP_\b',     # SQL Server 的擴展程序
        r';.*--',        # SQL Injection 常見模式
    ]
    
    # 允許的語句前綴（白名單）
    ALLOWED_PREFIXES = ['SELECT', 'WITH', 'EXPLAIN']  # 只允許讀取
    
    def validate(self, sql: str) -> dict:
        sql_clean = sql.strip().upper()
        
        # 檢查 1：是否以允許的語句開頭
        first_word = sql_clean.split()[0] if sql_clean.split() else ""
        if first_word not in self.ALLOWED_PREFIXES:
            return {"safe": False, "reason": f"只允許 SELECT/WITH/EXPLAIN，收到 '{first_word}'"}
        
        # 檢查 2：掃描危險關鍵字
        for pattern in self.DANGEROUS_KEYWORDS:
            if re.search(pattern, sql_clean):
                keyword = re.search(pattern, sql_clean).group()
                return {"safe": False, "reason": f"包含危險關鍵字：{keyword}"}
        
        # 檢查 3：多語句（; 注入）
        sql_stripped = re.sub(r"'[^']*'", "''", sql_clean)  # 移除字串內的 ;
        if sql_stripped.count(';') > 1:
            return {"safe": False, "reason": "不允許多語句（SQL 注入風險）"}
        
        return {"safe": True, "reason": "通過安全驗證"}

validator = SQLSafetyValidator()

test_sqls = [
    ("SELECT * FROM sales WHERE store_id='S001'",           "正常查詢"),
    ("DROP TABLE sales",                                    "危險：DROP"),
    ("SELECT * FROM sales; DELETE FROM sales",              "注入：多語句"),
    ("UPDATE sales SET unit_price=0 WHERE store_id='S001'", "危險：UPDATE"),
    ("WITH cte AS (SELECT * FROM sales) SELECT * FROM cte", "正常：CTE"),
    ("SELECT * FROM sales WHERE store_id='S001' -- AND 1=1","正常（含注釋）"),
]

print("SQL 安全驗證測試：")
print("=" * 60)
for sql, label in test_sqls:
    result = validator.validate(sql)
    icon = "✅" if result["safe"] else "🚫"
    print(f"  {icon} [{label}]")
    print(f"     SQL: {sql[:60]}")
    if not result["safe"]:
        print(f"     原因: {result['reason']}")
    print()

## 決策 2：資料隔離 — 為什麼不讓 LLM 自己加 WHERE？

**❌ 被拒絕：在 Prompt 裡說「查詢時要加 WHERE store_id='{user_store}'」**  
LLM 有機率「忘記」這個限制，特別是複雜查詢中。  
這是資料洩漏的風險，不能靠 LLM 記憶來保護。

**✅ 選擇：Query Rewriter — 程式碼層強制注入 WHERE 子句**

In [ ]:
class QueryRewriter:
    """
    在 SQL 執行前，強制注入 Row-Level Security 條件
    
    為什麼用 Query Rewriter，不用資料庫 RLS？
    → 如果用 BigQuery（不原生支援 RLS），或者多租戶邏輯複雜，
      程式碼層更靈活，且邏輯明確可測試
    → BigQuery 有 Row Access Policy，但設定複雜；
      簡單場景 Query Rewriter 更快
    
    什麼情況下改用資料庫 RLS？
    → 需要保護直接 SQL 存取（繞過應用層）
    → 多種應用共用同一資料庫
    → PostgreSQL/CloudSQL 有成熟的 RLS 支援
    """
    
    def inject_store_filter(self, sql: str, store_id: str) -> str:
        """強制在 sales 表的查詢中加入門市過濾"""
        sql_upper = sql.upper().strip()
        
        # 如果已經有 WHERE 子句
        if 'WHERE' in sql_upper:
            # 在現有 WHERE 後插入條件
            pattern = re.compile(r'(WHERE)', re.IGNORECASE)
            return pattern.sub(f"WHERE sales.store_id = '{store_id}' AND ", sql, count=1)
        
        # 如果沒有 WHERE，找 GROUP BY / ORDER BY / HAVING / LIMIT
        for keyword in ['GROUP BY', 'ORDER BY', 'HAVING', 'LIMIT']:
            if keyword in sql_upper:
                pattern = re.compile(r'(' + keyword + r')', re.IGNORECASE)
                return pattern.sub(f"WHERE sales.store_id = '{store_id}' {keyword}", sql, count=1)
        
        # 沒有任何 WHERE/GROUP 等，加在最後
        return sql.rstrip(';') + f" WHERE sales.store_id = '{store_id}'"


rewriter = QueryRewriter()

test_cases = [
    "SELECT * FROM sales",
    "SELECT product_name, SUM(quantity) FROM sales GROUP BY product_name",
    "SELECT * FROM sales WHERE product_name = '有機牛奶'",
    "SELECT * FROM sales ORDER BY sale_date DESC LIMIT 10",
]

print("Row-Level Security Query Rewriter：")
print("用戶門市：S001（台北信義店）")
print("=" * 60)
for sql in test_cases:
    rewritten = rewriter.inject_store_filter(sql, "S001")
    print(f"原始：{sql}")
    print(f"改寫：{rewritten}")
    print()

print("⭐ 重點：即使 LLM 沒有加 WHERE store_id，")
print("   Query Rewriter 也會強制加上——程式碼保證，不依賴 LLM")

## 決策 3：Schema 注入策略 — 為什麼不給完整 Schema？

In [ ]:
class SchemaInjector:
    """
    動態 Schema 注入（不是把所有 table 都丟給 LLM）
    
    為什麼不直接給完整 Schema？
    → 真實企業資料庫可能有 100+ 個表，完整 Schema 超過 10K tokens
    → 太多不相關的表會讓 LLM 混淆，降低 SQL 準確率
    → 成本：多 5K tokens per request × 5000 requests/day = 巨大浪費
    
    解法：
    1. 關鍵表 + 關係永遠包含（core schema）
    2. 根據 query 相關性動態選擇額外的表
    3. 加上少量範例（Few-shot SQL）
    """
    
    # 核心 Schema（永遠注入）
    CORE_SCHEMA = """
核心資料表：
  stores(store_id TEXT, store_name TEXT, region TEXT)
  sales(sale_id INT, store_id TEXT, product_name TEXT,
        quantity INT, unit_price REAL, sale_date TEXT)
  -- Revenue = quantity * unit_price
  -- sale_date 格式: 'YYYY-MM-DD'
"""
    
    # 可選 Schema（按 query 相關性動態加入）
    OPTIONAL_SCHEMAS = {
        "inventory": """
  inventory(product_id TEXT, store_id TEXT, stock_qty INT, reorder_level INT)
""",
        "customers": """
  customers(customer_id TEXT, name TEXT, membership TEXT, total_spending REAL)
""",
        "promotions": """
  promotions(promo_id TEXT, product_name TEXT, discount_pct REAL, start_date TEXT, end_date TEXT)
""",
    }
    
    # Few-shot 範例（示範正確的 SQL 格式）
    FEW_SHOT_EXAMPLES = """
SQL 範例：

問：本月銷售額是多少？
SQL: SELECT SUM(quantity * unit_price) as revenue
     FROM sales
     WHERE sale_date LIKE '2026-06%'

問：哪個商品賣最好？
SQL: SELECT product_name, SUM(quantity) as total_qty
     FROM sales
     GROUP BY product_name
     ORDER BY total_qty DESC
     LIMIT 5
"""
    
    def select_schema(self, query: str) -> str:
        """根據 query 動態選擇需要注入的 Schema"""
        schema = self.CORE_SCHEMA
        
        # 簡單關鍵字匹配決定需要哪些額外表
        if any(w in query for w in ["庫存", "補貨", "缺貨"]):
            schema += self.OPTIONAL_SCHEMAS["inventory"]
        if any(w in query for w in ["會員", "客戶", "消費者"]):
            schema += self.OPTIONAL_SCHEMAS["customers"]
        if any(w in query for w in ["折扣", "促銷", "優惠"]):
            schema += self.OPTIONAL_SCHEMAS["promotions"]
        
        schema += self.FEW_SHOT_EXAMPLES
        return schema


injector = SchemaInjector()

queries = [
    "本月銷售額是多少？",
    "庫存快不夠了的商品有哪些？",
    "哪個促銷活動效果最好？",
]

print("動態 Schema 注入：")
for q in queries:
    schema = injector.select_schema(q)
    tokens = len(schema.split()) * 4 // 3
    print(f"  Query: '{q}'")
    print(f"  Schema tokens: ~{tokens}（動態）vs ~1500（完整靜態）")
    if "inventory" in schema.lower():
        print(f"  ✅ 加入 inventory 表")
    print()

## 完整 Text-to-SQL Pipeline

In [ ]:
async def text_to_sql_pipeline(user_query: str, user_store_id: str,
                                 max_retries: int = 2) -> dict:
    """
    完整的 Text-to-SQL 管線：
    1. Schema 注入
    2. LLM 生成 SQL
    3. 安全驗證
    4. RLS 改寫（強制注入門市過濾）
    5. 執行
    6. 錯誤自動修正（最多 2 次）
    7. 結果轉自然語言
    """
    print(f"\n處理查詢：'{user_query}'（門市：{user_store_id}）")
    
    schema = injector.select_schema(user_query)
    
    for attempt in range(max_retries + 1):
        # Step 1: 生成 SQL
        if LLM_AVAILABLE:
            sql_prompt = f"""{schema}

生成 SQLite 查詢，只回覆 SQL，不要其他文字：
{user_query}"""
            resp = await llm.ainvoke([HumanMessage(content=sql_prompt)])
            sql = resp.content.strip().strip("```sql").strip("```").strip()
        else:
            # 模擬生成
            sql = "SELECT product_name, SUM(quantity * unit_price) as revenue FROM sales GROUP BY product_name ORDER BY revenue DESC LIMIT 5"
        
        print(f"  [Step {attempt+1}] 生成 SQL: {sql[:80]}..." if len(sql) > 80 else f"  [Step {attempt+1}] 生成 SQL: {sql}")
        
        # Step 2: 安全驗證
        safety = validator.validate(sql)
        if not safety["safe"]:
            print(f"  🚫 安全驗證失敗：{safety['reason']}")
            return {"success": False, "error": f"安全驗證失敗：{safety['reason']}"}
        
        # Step 3: 強制注入 RLS
        sql_with_rls = rewriter.inject_store_filter(sql, user_store_id)
        print(f"  ✅ RLS 注入完成（強制加 store_id='{user_store_id}'）")
        
        # Step 4: 執行
        try:
            cursor.execute(sql_with_rls)
            rows = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description]
            
            print(f"  ✅ 查詢成功，返回 {len(rows)} 筆資料")
            
            # Step 5: 結果轉自然語言
            result_str = json.dumps([dict(zip(columns, row)) for row in rows], ensure_ascii=False)
            if LLM_AVAILABLE:
                nl_prompt = f"""以下是查詢「{user_query}」的結果，請用繁體中文自然語言摘要（3句以內）：
{result_str[:500]}"""
                nl_resp = await llm.ainvoke([HumanMessage(content=nl_prompt)])
                answer = nl_resp.content
            else:
                answer = f"查詢結果：{result_str[:200]}"
            
            return {"success": True, "sql": sql_with_rls, "rows": rows,
                    "columns": columns, "answer": answer}
        
        except Exception as e:
            error_msg = str(e)
            print(f"  ❌ SQL 執行錯誤（attempt {attempt+1}）：{error_msg}")
            if attempt < max_retries:
                print(f"  🔄 自動修正中...")
                # 把錯誤回饋給 LLM 修正
                if LLM_AVAILABLE:
                    fix_prompt = f"""SQL 執行失敗：{error_msg}\n原 SQL：{sql}\n{schema}\n請修正 SQL，只回覆 SQL："""
                    fix_resp = await llm.ainvoke([HumanMessage(content=fix_prompt)])
                    sql = fix_resp.content.strip()
            else:
                return {"success": False, "error": error_msg}


# 執行測試
results = await text_to_sql_pipeline(
    user_query="這個月每個商品的銷售額是多少？",
    user_store_id="S001"
)

if results["success"]:
    print(f"\n✅ 最終答案：{results['answer']}")
    print(f"\n資料（僅限 S001 台北信義店）：")
    for row in results["rows"]:
        print(f"  {row}")

In [ ]:
print("""
FDE 面試：Text-to-SQL 架構決策
═══════════════════════════════════════════════════════

Q: 為什麼 SQL 安全要在程式碼層驗證，不靠 Prompt？
A: Prompt 是建議，程式碼是強制。安全不能靠建議。
   LLM 可能在複雜查詢中「忘記」限制，
   攻擊者可以用 Prompt Injection 繞過 System Prompt。
   程式碼層的 DROP/DELETE 關鍵字過濾是零信任設計。

Q: 資料隔離為什麼用 Query Rewriter 不用資料庫 RLS？
A: 取決於架構。
   Query Rewriter 優點：
   - 邏輯在應用層，清楚可測試
   - 適合 BigQuery（RLS 設定複雜）
   - 靈活，可以按業務邏輯動態決定 filter
   資料庫 RLS 優點：
   - 保護直接 SQL 存取（不依賴應用層）
   - PostgreSQL/CloudSQL 原生支援
   - 多應用共用資料庫時更安全

Q: 為什麼不把完整 Schema 給 LLM？
A: 三個原因：
   1. Token 成本：100 個表的 Schema ~10K tokens，每次請求都要付這個成本
   2. 準確率：太多不相關表格讓 LLM 混淆，SQL 正確率下降
   3. 安全：不讓 LLM 知道所有表格的存在（最小化洩漏）
   解法：動態 Schema 選擇，按 query 關鍵字決定注入哪些表。

Q: 錯誤恢復為什麼最多 2 次，不是 5 次？
A: 如果 2 次還沒修好，通常是 Schema 理解問題或 query 太複雜，
   繼續重試是浪費 Token。應該記錄這個 query 加入人工審核，
   分析是 Schema 需要改善，還是 query 需要分解。
═══════════════════════════════════════════════════════
""")